In [0]:
#------------------------------------ API KEYS, CLAVES DE ACCESO Y VARIABLES CLAVE ---------------------------------------------
import os
from dotenv import load_dotenv

load_dotenv()

storage_account_name = os.getenv("AZURE_STORAGE_ACCOUNT_NAME", "aybdwhstorage01")
storage_account_key = os.getenv("AZURE_STORAGE_ACCOUNT_KEY", "")


In [0]:
from pyspark.sql import SparkSession
from datetime import datetime, timedelta
import pyspark.sql.functions as F
from pyspark.sql.functions import date_sub, current_date, col, lit, when, concat, trim, coalesce, regexp_extract, regexp_replace, size, length, udf, sum as _sum, first, to_date, row_number, collect_list, struct, min as _min, array
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException
from pyspark.sql.types import StringType
import pandas as pd


In [0]:
# Iniciar sesión de Spark
spark = SparkSession.builder.appName("EventSales").getOrCreate()
config_key = f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net"
spark.conf.set(config_key, storage_account_key)

In [0]:
container_name = 'tuboleta'
localidades_gold = spark.read.parquet(f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/GOLD/SECUTIX/Training Data/Clustering de Localidades/")
#display(localidades_gold)

#localidades_gold = spark.read.parquet(f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/GOLD/SECUTIX/Training Data/Clustering de Localidades Sales/")
#display(localidades_gold)

In [0]:
# Eliminar eventos que contengan la palabra TEST o CANCELAD en el nombre del producto
registros_antes = localidades_gold.count()
localidades_gold = localidades_gold.filter(
    (~F.upper(col("product")).contains("TEST")) &
    (~F.upper(col("product")).contains("CANCELAD"))&
    (~F.upper(col("product")).contains("PARQUEA")) &
    (~F.upper(col("product")).contains("NO USAR")) &
    (col("performance_quota")>0)&
    (col("performance_quota").isNotNull()) &
    (col("net_sold_p_qty") <= col("dn_quota")) )

registros_filtro_texto = localidades_gold.count()
print(f"Registros antes: {registros_antes:,}")
print(f"Eliminados por texto/quota: {registros_antes - registros_filtro_texto:,}")

# Excluir performances 100% gratuitos/cortesías
# (donde TODOS sus registros tienen net_sold_c_qty > 0 o unit_amt_itx = 0)
from pyspark.sql.functions import count as _count

total_por_perf = localidades_gold.groupBy("t_performance_id").agg(_count("*").alias("total"))
condicion_por_perf = localidades_gold.filter(
    (col("net_sold_c_qty") > 0) | (col("unit_amt_itx") == 0)
).groupBy("t_performance_id").agg(_count("*").alias("condicion"))

performances_gratuitos = total_por_perf.join(condicion_por_perf, "t_performance_id", "inner").filter(
    col("total") == col("condicion")
).select("t_performance_id")

print(f"Performances 100% gratuitos/cortesías identificados: {performances_gratuitos.count():,}")

localidades_gold = localidades_gold.join(performances_gratuitos, "t_performance_id", "left_anti")
registros_sin_gratuitos = localidades_gold.count()
print(f"Eliminados por performances gratuitos: {registros_filtro_texto - registros_sin_gratuitos:,}")

# Eliminar duplicados exactos
localidades_gold = localidades_gold.dropDuplicates()
registros_despues = localidades_gold.count()
print(f"Duplicados eliminados: {registros_sin_gratuitos - registros_despues:,}")

print(f"\nRegistros finales: {registros_despues:,}")

Registros antes: 10,040,133
Eliminados por texto/quota: 158,397
Performances 100% gratuitos/cortesías identificados: 3,577
Eliminados por performances gratuitos: 187,298
Duplicados eliminados: 9,143,421

Registros finales: 551,017


In [0]:
# Crear columna contingent_cat: KILLS -> kills, FUERA DE CUPO -> ventas, el resto -> holds
localidades_gold = localidades_gold.withColumn(
    "contingent_cat",
    when(F.upper(col("contingent")).contains("KILL"), "kills")
    #.when((F.upper(col("contingent")) == "FUERA DE CUPO") | (F.upper(col("contingent")).contains("TAQUILLA")), "ventas")
    .otherwise("ventas")
)

print("=== DISTRIBUCIÓN DE CONTINGENT_CAT ===")
localidades_gold.groupBy("contingent_cat").count().orderBy("count", ascending=False).show()

# Filtrar registros con base_unit_amt_itx != 0
localidades_gold = localidades_gold.filter((col("contingent_cat") == "ventas") & ((col("base_unit_amt_itx") != 0) | (col("net_sold_c_qty") > 0)))
print(f"Registros después de filtrar base_unit_amt_itx != 0: {localidades_gold.count():,}")

# Agrupar por todas las columnas excepto net_sold_p_qty y net_sold_c_qty, y calcular la suma
group_cols = [c for c in localidades_gold.columns if c not in ("contingent_cat","contingent", "t_contingent_id", "dn_quota", "unit_amt_itx", "total_unit_amt_itx", "base_unit_amt_itx", "net_sold_p_qty", "net_sold_c_qty")]

localidades_gold_agg = localidades_gold.groupBy(group_cols).agg(
    F.max("dn_quota").alias("dn_quota"),
    F.avg("unit_amt_itx").alias("ave_unit_amt_itx"),
    F.avg("total_unit_amt_itx").alias("ave_total_unit_amt_itx"),
    F.avg("base_unit_amt_itx").alias("ave_base_unit_amt_itx"),
    F.median("unit_amt_itx").alias("med_unit_amt_itx"),
    F.median("total_unit_amt_itx").alias("med_total_unit_amt_itx"),
    F.median("base_unit_amt_itx").alias("med_base_unit_amt_itx"),
    F.sum("net_sold_p_qty").alias("net_sold_p_qty"),
    F.sum("net_sold_c_qty").alias("net_sold_c_qty")
)



=== DISTRIBUCIÓN DE CONTINGENT_CAT ===
+--------------+------+
|contingent_cat| count|
+--------------+------+
|        ventas|547716|
|         kills|  3301|
+--------------+------+

Registros después de filtrar base_unit_amt_itx != 0: 506,724


In [0]:
# Validar que la suma de dn_quota por t_performance_id == performance_quota
validacion = localidades_gold_agg.groupBy("t_performance_id").agg(
    F.sum("dn_quota").alias("suma_dn_quota"),
    F.first("performance_quota").alias("performance_quota"),
    F.first("product").alias("product"),
    F.first("site").alias("site")
).withColumn("coincide", col("suma_dn_quota") == col("performance_quota"))

total_perf = validacion.count()
coinciden = validacion.filter(col("coincide") == True).count()
no_coinciden = validacion.filter(col("coincide") == False).count()

print(f"Total performances: {total_perf:,}")
print(f"Coinciden (sum(dn_quota) == performance_quota): {coinciden:,} ({round(coinciden/total_perf*100,2)}%)")
print(f"No coinciden: {no_coinciden:,} ({round(no_coinciden/total_perf*100,2)}%)")

# Filtrar localidades_gold_agg para quedarnos solo con los performances que coinciden
performances_ok = validacion.filter(col("coincide") == True).select("t_performance_id")
localidades_gold_agg = localidades_gold_agg.join(performances_ok, "t_performance_id", "inner")

# Eliminar registros con sobreventa post-agregación (net_sold_p_qty > dn_quota)
localidades_gold_agg = localidades_gold_agg.filter(col("net_sold_p_qty") <= col("dn_quota"))

print(f"\nRegistros finales en localidades_gold_agg (solo performances que coinciden, sin sobreventa): {localidades_gold_agg.count():,}")

Total performances: 20,377
Coinciden (sum(dn_quota) == performance_quota): 18,651 (91.53%)
No coinciden: 1,726 (8.47%)

Registros finales en localidades_gold_agg (solo performances que coinciden, sin sobreventa): 34,030


In [0]:
# === ANÁLISIS DE CALIDAD DE DATOS EN localidades_gold_agg ===
print(f"Total registros: {localidades_gold_agg.count():,}")
print(f"Total columnas: {len(localidades_gold_agg.columns)}")

# 1. VALORES NULOS por columna
print("\n" + "="*60)
print("1. VALORES NULOS POR COLUMNA")
print("="*60)
null_counts = localidades_gold_agg.select(
    [F.sum(F.when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in localidades_gold_agg.columns]
)
display(null_counts)

# 2. VALORES CERO por columna numérica
print("\n" + "="*60)
print("2. VALORES CERO EN COLUMNAS NUMÉRICAS")
print("="*60)
numeric_cols = ["dn_quota", "performance_quota", "ave_unit_amt_itx", "ave_total_unit_amt_itx", 
                "ave_base_unit_amt_itx", "med_unit_amt_itx", "med_total_unit_amt_itx", 
                "med_base_unit_amt_itx", "net_sold_p_qty", "net_sold_c_qty"]

zero_counts = localidades_gold_agg.select(
    [F.sum(F.when(col(c) == 0, 1).otherwise(0)).alias(c) for c in numeric_cols]
)
display(zero_counts)

# 3. VALORES NEGATIVOS por columna numérica
print("\n" + "="*60)
print("3. VALORES NEGATIVOS EN COLUMNAS NUMÉRICAS")
print("="*60)
neg_counts = localidades_gold_agg.select(
    [F.sum(F.when(col(c) < 0, 1).otherwise(0)).alias(c) for c in numeric_cols]
)
display(neg_counts)

# 4. ESTADÍSTICAS DESCRIPTIVAS
print("\n" + "="*60)
print("4. ESTADÍSTICAS DESCRIPTIVAS")
print("="*60)
display(localidades_gold_agg.select(numeric_cols).summary())

# 5. INCONSISTENCIAS LÓGICAS
print("\n" + "="*60)
print("5. INCONSISTENCIAS LÓGICAS")
print("="*60)

# net_sold_p_qty negativo (¿devoluciones?)
neg_sold = localidades_gold_agg.filter(col("net_sold_p_qty") < 0).count()
print(f"Registros con net_sold_p_qty < 0: {neg_sold:,}")

# net_sold_c_qty negativo
neg_cortesias = localidades_gold_agg.filter(col("net_sold_c_qty") < 0).count()
print(f"Registros con net_sold_c_qty < 0: {neg_cortesias:,}")

# dn_quota = 0 (localidad sin cupo asignado)
dn_zero = localidades_gold_agg.filter(col("dn_quota") == 0).count()
print(f"Registros con dn_quota = 0: {dn_zero:,}")

# net_sold_p_qty > dn_quota (vendieron más que el cupo)
sobreventa = localidades_gold_agg.filter(col("net_sold_p_qty") > col("dn_quota")).count()
print(f"Registros con net_sold_p_qty > dn_quota (sobreventa): {sobreventa:,}")

# Precio promedio negativo
precio_neg = localidades_gold_agg.filter(col("ave_base_unit_amt_itx") < 0).count()
print(f"Registros con ave_base_unit_amt_itx < 0: {precio_neg:,}")

# product_code nulo
pc_null = localidades_gold_agg.filter(col("product_code").isNull()).count()
print(f"Registros con product_code nulo: {pc_null:,}")

# logical_seat_category nulo o vacío
lsc_null = localidades_gold_agg.filter((col("logical_seat_category").isNull()) | (col("logical_seat_category") == "")).count()
print(f"Registros con logical_seat_category nulo/vacío: {lsc_null:,}")

Total registros: 34,030
Total columnas: 19

1. VALORES NULOS POR COLUMNA


t_performance_id,t_product_id,product_code,product,product_family,product_date,site,t_site_id,performance_quota,logical_seat_category,dn_quota,ave_unit_amt_itx,ave_total_unit_amt_itx,ave_base_unit_amt_itx,med_unit_amt_itx,med_total_unit_amt_itx,med_base_unit_amt_itx,net_sold_p_qty,net_sold_c_qty
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0



2. VALORES CERO EN COLUMNAS NUMÉRICAS


dn_quota,performance_quota,ave_unit_amt_itx,ave_total_unit_amt_itx,ave_base_unit_amt_itx,med_unit_amt_itx,med_total_unit_amt_itx,med_base_unit_amt_itx,net_sold_p_qty,net_sold_c_qty
118,0,582,1854,1877,1080,2836,2846,1934,21961



3. VALORES NEGATIVOS EN COLUMNAS NUMÉRICAS


dn_quota,performance_quota,ave_unit_amt_itx,ave_total_unit_amt_itx,ave_base_unit_amt_itx,med_unit_amt_itx,med_total_unit_amt_itx,med_base_unit_amt_itx,net_sold_p_qty,net_sold_c_qty
0,0,0,146,134,0,138,132,146,0



4. ESTADÍSTICAS DESCRIPTIVAS


summary,dn_quota,performance_quota,ave_unit_amt_itx,ave_total_unit_amt_itx,ave_base_unit_amt_itx,med_unit_amt_itx,med_total_unit_amt_itx,med_base_unit_amt_itx,net_sold_p_qty,net_sold_c_qty
count,34030,34030,34030,34030,34030,34030,34030,34030,34030,34030
mean,612.7720246841022,3885.7843373493974,89471.31014667252,295950.463305462,58262.38628174921,95911.00770672936,240465.35843079633,81390.78766794005,30.663003232441962,25.992389068469
stddev,1610.92974069154,8575.46309746165,218917.25176884807,2994297.976153752,170126.1896068344,236048.6701708389,3260047.9026882257,212347.45435516912,106.59604679273286,108.63503381968653
min,0,1,0.0,-2.848755E8,-9000000.0,0.0,-2.848755E8,-9000000.0,-390.0,0.0
25%,75,120,12723.75,14250.0,6000.0,10680.0,10800.0,6300.0,6.0,0.0
50%,150,747,39371.42857142857,77583.33333333333,28333.333333333332,40000.0,64000.0,35350.0,13.0,0.0
75%,320,1521,93000.0,197142.85714285713,66186.11111111111,100000.0,170600.0,89000.0,24.0,7.0
max,50000,55011,1.536E7,2.2E8,1.536E7,1.536E7,2.8E8,1.536E7,3580.0,3382.0



5. INCONSISTENCIAS LÓGICAS
Registros con net_sold_p_qty < 0: 146
Registros con net_sold_c_qty < 0: 0
Registros con dn_quota = 0: 118
Registros con net_sold_p_qty > dn_quota (sobreventa): 0
Registros con ave_base_unit_amt_itx < 0: 134
Registros con product_code nulo: 0
Registros con logical_seat_category nulo/vacío: 0


In [0]:
caso = localidades_gold.filter(col("t_performance_id") == 10229457190725)

print(f"Performance: TEMPORADA 2024_2 - JUNIOR | ESTADIO METROPOLITANO")
print(f"performance_quota esperado: 45,514")
print(f"Total registros para este performance: {caso.count()}")
print(f"Suma de dn_quota: {caso.select(F.sum('dn_quota')).collect()[0][0]:,}")

print("\n=== DETALLE POR LOCALIDAD Y CONTINGENT ===")
display(caso)

Performance: TEMPORADA 2024_2 - JUNIOR | ESTADIO METROPOLITANO
performance_quota esperado: 45,514
Total registros para este performance: 16138
Suma de dn_quota: 125,510,963

=== DETALLE POR LOCALIDAD Y CONTINGENT ===


t_product_id,t_performance_id,product_code,product,contingent,t_contingent_id,product_family,product_date,site,t_site_id,performance_quota,logical_seat_category,dn_quota,unit_amt_itx,total_unit_amt_itx,base_unit_amt_itx,net_sold_p_qty,net_sold_c_qty
10229460112013,10229457190725,EONCE241,ONCE CALDAS 2024-1,FUERA DE CUPO,-1,COMPETICIÓN,2024-01-20,ESTADIO PALOGRANDE - MANIZALES,101356395873,28997,OCCIDENTAL LATERAL NORTE,1359,9845.5,9845.5,9845.5,0.0,0.0
10229460112013,10229457190725,EONCE241,ONCE CALDAS 2024-1,FUERA DE CUPO,-1,COMPETICIÓN,2024-01-20,ESTADIO PALOGRANDE - MANIZALES,101356395873,28997,ORIENTAL GENERAL,10894,22581.5,22581.5,22581.5,0.0,0.0
10229460112013,10229457190725,EONCE241,ONCE CALDAS 2024-1,FUERA DE CUPO,-1,COMPETICIÓN,2024-01-20,ESTADIO PALOGRANDE - MANIZALES,101356395873,28997,ORIENTAL GENERAL,10894,22581.5,-22581.5,-22581.5,0.0,0.0
10229460112013,10229457190725,EONCE241,ONCE CALDAS 2024-1,FUERA DE CUPO,-1,COMPETICIÓN,2024-01-20,ESTADIO PALOGRANDE - MANIZALES,101356395873,28997,OCCIDENTAL GENERAL,5629,30711.0,30711.0,30711.0,1.0,0.0
10229460112013,10229457190725,EONCE241,ONCE CALDAS 2024-1,FUERA DE CUPO,-1,COMPETICIÓN,2024-01-20,ESTADIO PALOGRANDE - MANIZALES,101356395873,28997,NORTE BARRAS,4847,13820.0,13820.0,13820.0,1.0,0.0
10229460112013,10229457190725,EONCE241,ONCE CALDAS 2024-1,FUERA DE CUPO,-1,COMPETICIÓN,2024-01-20,ESTADIO PALOGRANDE - MANIZALES,101356395873,28997,ORIENTAL GENERAL,10894,22581.5,22581.5,22581.5,1.0,0.0
10229460112013,10229457190725,EONCE241,ONCE CALDAS 2024-1,FUERA DE CUPO,-1,COMPETICIÓN,2024-01-20,ESTADIO PALOGRANDE - MANIZALES,101356395873,28997,OCCIDENTAL GENERAL,5629,23033.0,-23033.0,-23033.0,0.0,0.0
10229460112013,10229457190725,EONCE241,ONCE CALDAS 2024-1,FUERA DE CUPO,-1,COMPETICIÓN,2024-01-20,ESTADIO PALOGRANDE - MANIZALES,101356395873,28997,NORTE BARRAS,4847,17162.0,51486.0,17162.0,3.0,0.0
10229460112013,10229457190725,EONCE241,ONCE CALDAS 2024-1,FUERA DE CUPO,-1,COMPETICIÓN,2024-01-20,ESTADIO PALOGRANDE - MANIZALES,101356395873,28997,OCCIDENTAL GENERAL,5629,23033.0,23033.0,23033.0,1.0,0.0
10229460112013,10229457190725,EONCE241,ONCE CALDAS 2024-1,FUERA DE CUPO,-1,COMPETICIÓN,2024-01-20,ESTADIO PALOGRANDE - MANIZALES,101356395873,28997,SUR GENERAL,3646,10839.0,21678.0,10839.0,2.0,0.0


In [0]:
container_name = 'tuboleta'

output_path_gold = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/GOLD/SECUTIX/Training Data/Clustering de Localidades EDA/"

localidades_gold_agg.write.mode("overwrite").parquet(output_path_gold)